<a href="https://colab.research.google.com/github/Lianaulya/Practical-Statistics-for-Data-Scientists/blob/main/Chapter2_SamplingDistributions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Identitas Mahasiswa

| | |
|---|---|
| **Nama** | Muammar Yaldi Fadhil |
| **NIM** | 101032300019 |
| **Kelas** | TK-47-03 |
| **Mata Kuliah** | Machine Learning |

---

# Chapter 2 – Data and Sampling Distributions
**Book:** *Practical Statistics for Data Scientists* (O'Reilly)

Topics covered:
- Random Sampling & Sample Bias
- Sampling Distribution of a Statistic
- The Bootstrap
- Confidence Intervals
- Normal Distribution
- Long-Tailed Distributions
- Student's t-Distribution
- Binomial Distribution
- Poisson Distribution


In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

BASE_DATA = "https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data"
print("Libraries loaded successfully!")

## 1. Random Sampling

A **simple random sample** gives every element an equal probability of selection.
Key concern: **sample bias** – when some members are systematically more/less likely to be selected.

**Stratified sampling** divides the population into sub-groups (strata) and samples from each.
**Cluster sampling** randomly selects entire groups (clusters).

In [ ]:
# Loans income data
loans_income = pd.read_csv(f"{BASE_DATA}/loans_income.csv", squeeze=True)

print(f"Population size: {len(loans_income):,}")
print(f"Population Mean: ${loans_income.mean():,.2f}")
print(f"Population Std:  ${loans_income.std():,.2f}")

In [ ]:
# Take a random sample of 1000
np.random.seed(42)
sample = loans_income.sample(1000)
print(f"Sample Mean:  ${sample.mean():,.2f}")
print(f"Sample Std:   ${sample.std():,.2f}")
print(f"Error in mean: ${abs(sample.mean() - loans_income.mean()):,.2f}")

## 2. Sampling Distribution of a Statistic

The **sampling distribution** of a statistic (e.g., the sample mean) describes how that
statistic varies from sample to sample.

**Central Limit Theorem (CLT)**: As sample size $n$ increases, the sampling distribution
of the mean approaches a normal distribution:

$$\bar{X} \sim \mathcal{N}\!\left(\mu,\; \frac{\sigma^2}{n}\right)$$

**Standard Error** of the mean: $SE = \dfrac{\sigma}{\sqrt{n}}$

In [ ]:
# Demonstrate CLT: sample means from 1000 repetitions
np.random.seed(42)
sample_means_5  = [loans_income.sample(5).mean() for _ in range(1000)]
sample_means_20 = [loans_income.sample(20).mean() for _ in range(1000)]
sample_means_100= [loans_income.sample(100).mean() for _ in range(1000)]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, means, n in zip(axes, [sample_means_5, sample_means_20, sample_means_100], [5, 20, 100]):
    pd.Series(means).hist(bins=30, ax=ax, color="steelblue", edgecolor="black")
    ax.set_title(f"n = {n} | SE = ${np.std(means):,.0f}")
    ax.set_xlabel("Sample Mean Income ($)")
axes[0].set_ylabel("Frequency")
fig.suptitle("Central Limit Theorem – Sampling Distribution of the Mean", y=1.02)
plt.tight_layout()
plt.show()

## 3. The Bootstrap

The **bootstrap** is a resampling technique that estimates the sampling distribution
of a statistic by repeatedly resampling *with replacement* from the original data:

1. Draw $B$ bootstrap samples of size $n$ (with replacement)
2. Compute the statistic $\hat{\theta}^*_b$ for each sample
3. The distribution of $\hat{\theta}^*_b$ approximates the sampling distribution

Bootstrap standard error: $SE_{boot} = \sqrt{\dfrac{1}{B-1}\sum_{b=1}^B (\hat{\theta}^*_b - \bar{\theta}^*)^2}$

In [ ]:
# Bootstrap estimation of the median income
np.random.seed(42)
sample_data = loans_income.sample(1000)

n_boot = 1000
bootstrap_medians = [
    sample_data.sample(len(sample_data), replace=True).median()
    for _ in range(n_boot)
]

boot_se = np.std(bootstrap_medians)
boot_ci_lo = np.percentile(bootstrap_medians, 2.5)
boot_ci_hi = np.percentile(bootstrap_medians, 97.5)

print(f"Sample Median:        ${sample_data.median():,.2f}")
print(f"Bootstrap SE:         ${boot_se:,.2f}")
print(f"95% CI:               [${boot_ci_lo:,.2f}, ${boot_ci_hi:,.2f}]")

fig, ax = plt.subplots(figsize=(8, 4))
pd.Series(bootstrap_medians).hist(bins=30, ax=ax, color="steelblue", edgecolor="black")
ax.axvline(boot_ci_lo, color="red", linestyle="--", label="2.5th pctile")
ax.axvline(boot_ci_hi, color="red", linestyle="-.", label="97.5th pctile")
ax.set_title("Bootstrap Distribution of Median Income")
ax.set_xlabel("Median Income ($)")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Confidence Intervals

A **confidence interval** (CI) provides a range of plausible values for the population parameter.
A 95% CI means: if we repeated the sampling process many times, 95% of the resulting CIs
would contain the true parameter.

For large samples using the CLT:

$$\text{95% CI} = \bar{x} \pm 1.96 \cdot \frac{s}{\sqrt{n}}$$

In [ ]:
# Confidence intervals via bootstrap (already computed above)
# Let us also compute the formula-based CI for the mean

n = len(sample_data)
x_bar = sample_data.mean()
se = sample_data.std() / np.sqrt(n)
ci_low  = x_bar - 1.96 * se
ci_high = x_bar + 1.96 * se

print(f"Formula-based 95% CI for Mean Income:")
print(f"  Mean:  ${x_bar:,.2f}")
print(f"  SE:    ${se:,.2f}")
print(f"  Lower: ${ci_low:,.2f}")
print(f"  Upper: ${ci_high:,.2f}")

## 5. Normal Distribution

The **normal (Gaussian) distribution** is defined by:

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{1}{2}\left(\frac{x-\mu}{\sigma}\right)^2}$$

Key properties:
- Symmetric around $\mu$
- 68% of data within $\mu \pm 1\sigma$
- 95% within $\mu \pm 2\sigma$
- 99.7% within $\mu \pm 3\sigma$ (empirical rule)

**Standardization / Z-score**: $z = \dfrac{x - \mu}{\sigma}$

In [ ]:
# QQ-plot: check normality of income sample
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sample_data.hist(bins=30, ax=axes[0], color="steelblue", edgecolor="black", density=True)
x = np.linspace(sample_data.min(), sample_data.max(), 200)
axes[0].plot(x, stats.norm.pdf(x, sample_data.mean(), sample_data.std()),
             "r-", linewidth=2, label="Normal PDF")
axes[0].set_title("Loan Income – Histogram vs. Normal")
axes[0].set_xlabel("Income ($)")
axes[0].legend()

# QQ plot
stats.probplot(sample_data, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q Plot")

plt.tight_layout()
plt.show()

## 6. Long-Tailed Distributions

Not all data is normally distributed. **Long-tailed** (or heavy-tailed) distributions
have more probability in the tails than the normal distribution.

Examples: income, stock returns, city populations (power-law tails).
**Skewness** measures the asymmetry; **kurtosis** measures tail heaviness.

In [ ]:
# S&P 500 returns — check for fat tails
sp500_px = pd.read_csv(f"{BASE_DATA}/sp500_data.csv.gz", index_col=0)
sp500_returns = sp500_px.pct_change().dropna()

# Take Apple as example
aapl = sp500_returns["AAPL"]
print(f"AAPL Returns — Skewness: {aapl.skew():.4f}, Kurtosis: {aapl.kurtosis():.4f}")

fig, ax = plt.subplots(figsize=(9, 4))
aapl.hist(bins=60, ax=ax, color="steelblue", edgecolor="black", density=True)
x = np.linspace(aapl.min(), aapl.max(), 200)
ax.plot(x, stats.norm.pdf(x, aapl.mean(), aapl.std()), "r-", linewidth=2, label="Normal")
ax.set_title("AAPL Daily Returns — Heavy Tails vs. Normal")
ax.set_xlabel("Daily Return")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Student's t-Distribution

When the population variance is **unknown** and the sample size is small, we use Student's **t-distribution**:

$$t = \frac{\bar{X} - \mu}{s/\sqrt{n}}$$

with $n-1$ degrees of freedom. As $n \to \infty$, the t-distribution converges to the normal.

In [ ]:
# Compare t vs. normal for different degrees of freedom
x = np.linspace(-4, 4, 200)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x, stats.norm.pdf(x), "k-", linewidth=2, label="Normal")
for df, col in [(1, "red"), (3, "orange"), (10, "steelblue"), (30, "green")]:
    ax.plot(x, stats.t.pdf(x, df), linestyle="--", label=f"t (df={df})", color=col)
ax.set_title("t-Distribution vs. Normal — Heavier Tails for Small df")
ax.set_xlabel("Value")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Binomial Distribution

The **binomial distribution** counts successes in $n$ independent Bernoulli trials,
each with success probability $p$:

$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$$

Mean: $\mu = np$ &emsp; Variance: $\sigma^2 = np(1-p)$

In [ ]:
# Binomial example: click-through rate
n, p = 200, 0.05  # 200 visitors, 5% click rate
k = np.arange(0, 26)
probs = stats.binom.pmf(k, n, p)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(k, probs, color="steelblue", edgecolor="black", alpha=0.7)
ax.set_title(f"Binomial Distribution — n={n}, p={p}")
ax.set_xlabel("Number of Successes")
ax.set_ylabel("Probability")
ax.axvline(n * p, color="red", linestyle="--", label=f"Mean = {n*p}")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Poisson Distribution

The **Poisson distribution** models the number of events in a fixed time interval,
given a known average rate $\lambda$:

$$P(X = k) = \frac{\lambda^k e^{-\lambda}}{k!}$$

Mean $= \lambda$, Variance $= \lambda$

In [ ]:
# Poisson example: server requests per second
lambda_ = 7  # average 7 requests per second
k = np.arange(0, 21)
probs_pois = stats.poisson.pmf(k, lambda_)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(k, probs_pois, color="coral", edgecolor="black", alpha=0.8)
ax.set_title(f"Poisson Distribution — λ = {lambda_} events/sec")
ax.set_xlabel("Number of Events (k)")
ax.set_ylabel("Probability")
ax.axvline(lambda_, color="navy", linestyle="--", label=f"Mean = {lambda_}")
ax.legend()
plt.tight_layout()
plt.show()

## Summary

| Concept | Key Point |
|---------|----------|
| CLT | Sample mean → Normal as $n$ grows |
| SE | $\sigma / \sqrt{n}$ — precision of the sample mean |
| Bootstrap | Resampling to estimate sampling distribution |
| 95% CI | $\bar{x} \pm 1.96 \cdot SE$ |
| t-dist | Heavier tails than Normal; used when $\sigma$ unknown |
| Binomial | Count of successes in $n$ Bernoulli trials |
| Poisson | Count of events in fixed interval with rate $\lambda$ |
